# Unified Logistics Data Platform - Data Exploration

This notebook demonstrates the data flowing through the logistics platform:
- **Fleet Telematics**: Vehicle GPS and trip data
- **Shipment Tracking**: Package journey through hub network
- **Last-Mile Delivery**: Delivery agent performance

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Try to import DuckDB for reading Delta/Parquet
try:
    import duckdb
    print("Using DuckDB for data access")
except ImportError:
    print("Install duckdb: pip install duckdb")

# Configure display
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')

# Data paths
DATA_DIR = Path('../data')
BRONZE_DIR = DATA_DIR / 'bronze'
SILVER_DIR = DATA_DIR / 'silver'
GOLD_DIR = DATA_DIR / 'gold'

## Helper Functions

In [ ]:
def read_delta_table(path: str) -> pd.DataFrame:
    """Read Delta/Parquet table using DuckDB."""
    try:
        return duckdb.query(f"SELECT * FROM read_parquet('{path}/**/*.parquet')").df()
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return pd.DataFrame()

def summarize_df(df: pd.DataFrame, name: str):
    """Print summary statistics for a DataFrame."""
    print(f"\n{'='*60}")
    print(f" {name}")
    print(f"{'='*60}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"\nColumn Types:")
    print(df.dtypes.value_counts())
    print(f"\nSample:")
    display(df.head())

---
# 1. Fleet Telematics

## 1.1 Vehicle Positions (Bronze)

In [ ]:
# Load vehicle positions
vehicle_positions = read_delta_table(str(BRONZE_DIR / 'vehicle_positions'))
summarize_df(vehicle_positions, 'Vehicle Positions (Bronze)')

In [ ]:
# Plot vehicle positions on map
if len(vehicle_positions) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Sample for visualization
    sample = vehicle_positions.sample(min(5000, len(vehicle_positions)))
    
    scatter = ax.scatter(
        sample['longitude'], 
        sample['latitude'],
        c=sample['speed_kmh'],
        cmap='RdYlGn_r',
        alpha=0.5,
        s=5
    )
    
    plt.colorbar(scatter, label='Speed (km/h)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Vehicle Positions Across India')
    plt.tight_layout()
    plt.show()

In [ ]:
# Speed distribution by vehicle type
if len(vehicle_positions) > 0 and 'vehicle_type' in vehicle_positions.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    vehicle_positions.boxplot(
        column='speed_kmh',
        by='vehicle_type',
        ax=ax
    )
    
    ax.set_title('Speed Distribution by Vehicle Type')
    ax.set_xlabel('Vehicle Type')
    ax.set_ylabel('Speed (km/h)')
    plt.suptitle('')  # Remove automatic title
    plt.tight_layout()
    plt.show()

## 1.2 Reconstructed Trips (Silver)

In [ ]:
# Load trips
trips = read_delta_table(str(SILVER_DIR / 'fleet' / 'trips'))
summarize_df(trips, 'Reconstructed Trips (Silver)')

In [ ]:
# Trip metrics summary
if len(trips) > 0:
    print("\nTrip Statistics:")
    print(f"  Total Trips: {len(trips):,}")
    print(f"  Avg Distance: {trips['total_distance_km'].mean():.1f} km")
    print(f"  Avg Duration: {trips['trip_duration_minutes'].mean():.1f} min")
    print(f"  Avg Speed: {trips['avg_speed_kmh'].mean():.1f} km/h")
    
    # Trip type distribution
    if 'trip_type' in trips.columns:
        print("\nTrip Types:")
        print(trips['trip_type'].value_counts())

---
# 2. Shipment Tracking

## 2.1 Shipment Events (Bronze)

In [ ]:
# Load shipment events
shipment_events = read_delta_table(str(BRONZE_DIR / 'shipment_events'))
summarize_df(shipment_events, 'Shipment Events (Bronze)')

In [ ]:
# Event type distribution
if len(shipment_events) > 0 and 'event_type' in shipment_events.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    event_counts = shipment_events['event_type'].value_counts()
    colors = plt.cm.Spectral(np.linspace(0, 1, len(event_counts)))
    
    event_counts.plot(kind='barh', ax=ax, color=colors)
    
    ax.set_xlabel('Count')
    ax.set_title('Shipment Event Type Distribution')
    plt.tight_layout()
    plt.show()

In [ ]:
# Hub activity
if len(shipment_events) > 0 and 'hub_name' in shipment_events.columns:
    print("\nHub Activity:")
    hub_activity = shipment_events.groupby('hub_name').agg(
        event_count=('event_id', 'count'),
        unique_shipments=('shipment_id', 'nunique')
    ).sort_values('event_count', ascending=False)
    display(hub_activity)

## 2.2 Shipment Journeys (Silver)

In [ ]:
# Load journeys
journeys = read_delta_table(str(SILVER_DIR / 'shipment' / 'journeys'))
summarize_df(journeys, 'Shipment Journeys (Silver)')

In [ ]:
# Journey outcome analysis
if len(journeys) > 0 and 'journey_outcome' in journeys.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Outcome distribution
    outcome_counts = journeys['journey_outcome'].value_counts()
    colors = {'DELIVERED': 'green', 'IN_TRANSIT': 'blue', 'FAILED': 'red', 
              'RETURNED': 'orange', 'STUCK': 'gray'}
    bar_colors = [colors.get(x, 'gray') for x in outcome_counts.index]
    
    outcome_counts.plot(kind='bar', ax=axes[0], color=bar_colors)
    axes[0].set_title('Journey Outcomes')
    axes[0].set_xlabel('Outcome')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # SLA status
    if 'sla_status' in journeys.columns:
        sla_counts = journeys['sla_status'].value_counts()
        sla_colors = {'MET': 'green', 'BREACHED': 'red', 'IN_PROGRESS': 'blue', 'FAILED': 'gray'}
        pie_colors = [sla_colors.get(x, 'gray') for x in sla_counts.index]
        
        axes[1].pie(sla_counts, labels=sla_counts.index, colors=pie_colors,
                    autopct='%1.1f%%', startangle=90)
        axes[1].set_title('SLA Compliance')
    
    plt.tight_layout()
    plt.show()

---
# 3. Last-Mile Delivery

## 3.1 Delivery Events (Bronze)

In [ ]:
# Load delivery events
delivery_events = read_delta_table(str(BRONZE_DIR / 'delivery_events'))
summarize_df(delivery_events, 'Delivery Events (Bronze)')

In [ ]:
# Delivery success analysis
if len(delivery_events) > 0 and 'event_type' in delivery_events.columns:
    delivery_outcomes = delivery_events['event_type'].value_counts()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = {'DELIVERED': 'green', 'DELIVERY_ATTEMPTED': 'orange', 'DELIVERY_FAILED': 'red'}
    bar_colors = [colors.get(x, 'gray') for x in delivery_outcomes.index]
    
    delivery_outcomes.plot(kind='bar', ax=ax, color=bar_colors)
    ax.set_title('Delivery Outcomes')
    ax.set_xlabel('Event Type')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Success rate
    if 'DELIVERED' in delivery_outcomes.index:
        total_attempts = delivery_outcomes.sum()
        success_rate = delivery_outcomes.get('DELIVERED', 0) / total_attempts * 100
        print(f"\nDelivery Success Rate: {success_rate:.1f}%")

In [ ]:
# Customer ratings distribution
if len(delivery_events) > 0 and 'customer_rating' in delivery_events.columns:
    ratings = delivery_events['customer_rating'].dropna()
    
    if len(ratings) > 0:
        fig, ax = plt.subplots(figsize=(8, 5))
        ratings.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title('Customer Ratings Distribution')
        ax.set_xlabel('Rating')
        ax.set_ylabel('Count')
        plt.tight_layout()
        plt.show()
        
        print(f"\nAverage Rating: {ratings.mean():.2f}")

## 3.2 Agent Shifts (Silver)

In [ ]:
# Load agent shifts
agent_shifts = read_delta_table(str(SILVER_DIR / 'delivery' / 'agent_shifts'))
summarize_df(agent_shifts, 'Agent Shifts (Silver)')

In [ ]:
# Agent performance analysis
if len(agent_shifts) > 0 and 'performance_tier' in agent_shifts.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Performance tier distribution
    tier_counts = agent_shifts['performance_tier'].value_counts()
    tier_colors = {
        'TOP_PERFORMER': 'gold',
        'GOOD': 'green',
        'AVERAGE': 'blue',
        'BELOW_AVERAGE': 'red'
    }
    colors = [tier_colors.get(x, 'gray') for x in tier_counts.index]
    
    tier_counts.plot(kind='bar', ax=axes[0], color=colors)
    axes[0].set_title('Agent Performance Tiers')
    axes[0].set_xlabel('Performance Tier')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Deliveries per hour distribution
    if 'deliveries_per_hour' in agent_shifts.columns:
        agent_shifts['deliveries_per_hour'].hist(ax=axes[1], bins=20, color='steelblue', edgecolor='white')
        axes[1].set_title('Deliveries per Hour Distribution')
        axes[1].set_xlabel('Deliveries per Hour')
        axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

---
# 4. Data Quality Summary

In [ ]:
# Summary of all data
print("\n" + "="*60)
print(" DATA SUMMARY")
print("="*60)

data_summary = {
    'Table': [],
    'Layer': [],
    'Row Count': [],
    'Columns': []
}

datasets = [
    ('Vehicle Positions', 'Bronze', vehicle_positions),
    ('Shipment Events', 'Bronze', shipment_events),
    ('Delivery Events', 'Bronze', delivery_events),
    ('Trips', 'Silver', trips),
    ('Journeys', 'Silver', journeys),
    ('Agent Shifts', 'Silver', agent_shifts),
]

for name, layer, df in datasets:
    data_summary['Table'].append(name)
    data_summary['Layer'].append(layer)
    data_summary['Row Count'].append(f"{len(df):,}" if len(df) > 0 else "No data")
    data_summary['Columns'].append(len(df.columns) if len(df) > 0 else 0)

summary_df = pd.DataFrame(data_summary)
display(summary_df)

---
## Next Steps

1. **Run dbt models** to create mart tables with business metrics
2. **Explore data quality reports** in `data/quality_reports/`
3. **Check Airflow UI** at http://localhost:8080 to monitor pipeline
4. **View dbt docs** with `make dbt-docs` for data lineage